In [1]:
# Scenario: "The Environmental Policy Compliance Assistant"
# Background
# You are part of the Sustainability & Environmental Compliance Team at a global manufacturing company.
# New government regulations on carbon emissions, waste disposal, and renewable energy adoption have just been released.
# The company must ensure compliance to avoid fines and reputational damage.

# Challenge
# The company uploads a PDF of the environmental regulation into the Compliance Assistant (Gradio + Chroma + LLM app).
# Your task is to:
# - Process the regulation document so the assistant can store and understand it.
# - Ask compliance-related questions about emission limits, waste management rules, and renewable energy targets.
# - Generate clear, actionable answers that can guide engineers, sustainability officers, and executives.

# Roles
# - Learner (You): Environmental compliance officer using the assistant.
# - Assistant (The RAG App): Provides answers strictly based on uploaded environmental regulations.
# - Stakeholders: Plant managers, sustainability officers, and executives who need concise compliance guidance.

# 🔄 Flow of the Scenario
# - Upload Environmental Regulation PDF
# Example: “National Carbon Emissions Act 2026”.
# - System Processes Document
# - Splits into chunks.
# - Embeds into vector database.
# - Stores for retrieval.
# - Ask Questions
# - “What is the maximum carbon emission allowed per factory per year?”
# - “What penalties apply if hazardous waste is not disposed of properly?”
# - “What renewable energy targets must we meet by 2030?”
# - Assistant Responds
# - Retrieves relevant chunks.
# - Generates compliance-focused answers.
# - Provides short, clear guidance.
# - Outcome
# - Learners practice extracting environmental obligations.
# - Managers receive summarized compliance insights.
# - Executives gain confidence in sustainability strategy alignment.

# 🎯 Training Objective
# This scenario helps learners:
# - Understand how RAG systems can support environmental compliance.
# - Practice formulating precise queries to extract obligations.
# - Experience how AI can simplify complex environmental regulations into actionable steps.

# 👉 Would you like me to also draft a sample regulation PDF text (like the healthcare one I created earlier) for this environmental context, so you can upload it into your assistant and simulate queries?

In [ ]:
# ==========================================================
# ENVIRONMENTAL POLICY COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + HUGGINGFACE LLM
# ==========================================================

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# ----------------------------------------------------------
# STEP 1 — Load Language Model
# ----------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# ----------------------------------------------------------
# STEP 2 — Load Embedding Model
# ----------------------------------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 3 — Initialize Vector Database
# ----------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("environmental_policy_docs")
except:
    pass

collection = client.create_collection("environmental_policy_docs")


# ----------------------------------------------------------
# STEP 4 — Chunking Function
# ----------------------------------------------------------

def chunk_text(text, chunk_size=600, overlap=80):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# ----------------------------------------------------------
# STEP 5 — Process Uploaded Regulation PDF
# ----------------------------------------------------------

def process_pdf(file):

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text

    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Environmental regulation processed. {len(chunks)} sections indexed."


# ----------------------------------------------------------
# STEP 6 — Retriever
# ----------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 7 — Generate Compliance Answer
# ----------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are an Environmental Policy Compliance Assistant.

Use ONLY the regulation context below to answer the question.

Provide clear and actionable guidance for engineers,
sustainability officers, and executives.

Context:
{context}

Question: {query}

Answer:
"""

    response = llm(
        prompt,
        max_length=220,
        temperature=0.2
    )

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 8 — Chat Function
# ----------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a compliance question."

    return answer_question(question)


# ----------------------------------------------------------
# STEP 9 — Gradio Interface
# ----------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 🌍 Environmental Policy Compliance Assistant")

    gr.Markdown("""
Upload an environmental regulation document and ask questions about:

• Carbon emission limits  
• Waste disposal rules  
• Renewable energy targets  
• Environmental compliance penalties
""")

    pdf_input = gr.File(label="Upload Environmental Regulation PDF")

    upload_button = gr.Button("Process Regulation Document")

    status = gr.Textbox(label="System Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Compliance Question"
    )

    answer_box = gr.Textbox(
        label="Compliance Guidance",
        lines=12
    )

    ask_button = gr.Button("Get Answer")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# ----------------------------------------------------------
# STEP 10 — Launch App
# ----------------------------------------------------------

demo.launch()

Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Passing `generation_config` together with generation-related arguments=({'max_length', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=220) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
